# Inventory Optimization Training Pipeline

This notebook trains the HOBBIES detrended LightGBM models used by the inventory simulator. Run the cells from top to bottom. The pipeline supports Kaggle input paths and local execution.

**Outputs:** processed features in `Data/`, trained models in `Models/`, and evaluation metadata beside the processed data.

In [ ]:
import os, gc, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
from lightgbm.callback import EarlyStopException
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Paths are portable: run on Kaggle or locally from the pipeline/ folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'pipeline':
    PROJECT_ROOT = PROJECT_ROOT.parent

IS_KAGGLE = Path('/kaggle/input').exists()
M5_INPUT_DIR = Path(os.environ.get(
    'M5_INPUT_DIR',
    '/kaggle/input/competitions/m5-forecasting-accuracy' if IS_KAGGLE
    else str(PROJECT_ROOT / 'raw_data' / 'm5-forecasting-accuracy')
))
OUTPUT_ROOT = Path('/kaggle/working/inventory_optimization' if IS_KAGGLE else PROJECT_ROOT)
DATA_DIR = OUTPUT_ROOT / 'Data'
MODEL_DIR_DT = OUTPUT_ROOT / 'Models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR_DT.mkdir(parents=True, exist_ok=True)
OUT = OUTPUT_ROOT

print('M5 input:', M5_INPUT_DIR)
print('Processed data output:', DATA_DIR)
print('Model output:', MODEL_DIR_DT)

ID_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
START_TRAIN = 700
TRAIN_END = 1829
CV_WINDOWS = {'cv1': (1830, 1857), 'cv2': (1858, 1885), 'cv3': (1886, 1913)}
PUBLIC = (1914, 1941)
TEST_END = 1941
VAL_NAMES = list(CV_WINDOWS.keys())
SHIFT = 28
lags = [1, 2, 3, 6, 12, 24, 36]
ROLL_WINDOWS = [7, 14, 30, 60, 180]
PATIENCE = 100
MAX_ROUNDS = 3000
LAMBDA_STD = 1.5

PARAMS_DT = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'subsample': 0.5,
    'subsample_freq': 1,
    'learning_rate': 0.015,
    'num_leaves': 2**8 - 1,
    'min_data_in_leaf': 2**8 - 1,
    'feature_fraction': 0.5,
    'max_bin': 100,
    'boost_from_average': False,
    'verbose': -1,
}

## 1. Configure paths and experiment parameters

This step detects Kaggle automatically. On Kaggle, artifacts are written to `/kaggle/working/inventory_optimization/`; locally, they are written to the project root. Set the `M5_INPUT_DIR` environment variable when the raw M5 files are stored elsewhere.

In [ ]:
day_cols = [f'd_{i}' for i in range(1, TEST_END + 1)]

sales = pd.read_csv(M5_INPUT_DIR / 'sales_train_evaluation.csv')
sales = sales[sales['cat_id'] == 'HOBBIES'][ID_COLS + day_cols].reset_index(drop=True)

calendar = pd.read_csv(M5_INPUT_DIR / 'calendar.csv')
calendar = calendar[calendar['d'].isin(day_cols)].reset_index(drop=True)
calendar['d'] = calendar['d'].str.split('_').str[1].astype(np.int16)
calendar = calendar.drop(columns=['date'])

prices = pd.read_csv(M5_INPUT_DIR / 'sell_prices.csv')
prices = prices[prices['item_id'].isin(sales['item_id'].unique())].reset_index(drop=True)

df = pd.melt(sales, id_vars=ID_COLS, var_name='d', value_name='sold')
df['d'] = df['d'].str.split('_').str[1].astype(np.int16)
df = df.merge(calendar, on='d', how='left')
df = df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
del sales, prices
gc.collect()

df['id_str'] = df['id'].astype(str)
print(df.shape)

## 2. Load and reshape the M5 HOBBIES data

Read `sales_train_evaluation.csv`, `calendar.csv`, and `sell_prices.csv`; keep the HOBBIES category; convert the sales table to daily long format; and merge calendar and price features.

In [ ]:
name_maps = {}
for c in ID_COLS:
    cat = df[c].astype('category')
    name_maps[c] = dict(enumerate(cat.cat.categories))
    df[c] = cat.cat.codes.astype(np.int32)

for c in df.select_dtypes(include=['object', 'category']).columns:
    if c != 'id_str':
        df[c] = df[c].astype('category').cat.codes.astype(np.int16)

df = df.sort_values(ID_COLS + ['d']).reset_index(drop=True)

groups = list(df[['store_id', 'dept_id']].drop_duplicates()
              .sort_values(['store_id', 'dept_id'])
              .itertuples(index=False, name=None))
print('Số nhóm (store, dept):', len(groups))

## 3. Encode identifiers and prepare series groups

Encode categorical identifiers consistently, preserve `name_maps.json` for interpretation, sort each series by time, and identify the 20 `(store, department)` model groups.

In [ ]:
sold_wide = (df.pivot(index='id_str', columns='d', values='sold')
             .reindex(columns=range(1, TEST_END + 1)))
ids = sold_wide.index.to_numpy()
S_sold = sold_wide.to_numpy(dtype=np.float64)
print(S_sold.shape)

## 4. Build a wide sales matrix for evaluation

Create the item-by-day matrix used for RMSSE calculations. This matrix is evaluation-only and is not used as a model feature table.

In [ ]:
BASE_W = 14   # đổi thành 14 hoặc 60 để so sánh

# Mức bán trung bình trên mỗi chuỗi, theo (store, dept, ngày)
grp_daily = (df.groupby(['store_id', 'dept_id', 'd'])['sold']
             .mean().rename('gm').reset_index()
             .sort_values(['store_id', 'dept_id', 'd']))

# Trung bình BASE_W ngày, lệch SHIFT ngày: chỉ dùng dữ liệu đã biết tại ngày d
grp_daily['trend'] = (grp_daily.groupby(['store_id', 'dept_id'])['gm']
                      .transform(lambda x: x.shift(SHIFT).rolling(BASE_W, min_periods=1).mean()))

df = df.merge(grp_daily[['store_id', 'dept_id', 'd', 'trend']],
              on=['store_id', 'dept_id', 'd'], how='left')
df = df.sort_values(ID_COLS + ['d']).reset_index(drop=True)

df['trend'] = df['trend'].fillna(0.0)
df['y'] = df['sold'] - df['trend']

print('Tỉ lệ dòng có base NaN:', round(df['trend'].isna().mean(), 4))
print('Tỉ lệ dòng test có base < 0:',
      round((df.loc[df['d'] > TRAIN_END, 'trend'] < 0).mean(), 4))

## 5. Create the detrending baseline

Estimate the group-level trend using only information shifted by 28 days. The detrended target is `y = sold - trend`. Change `BASE_W` only when intentionally running a new experiment.

In [ ]:
keys = [df[c] for c in ID_COLS]
shifted = df.groupby(ID_COLS, observed=True)['y'].shift(SHIFT)
gs = shifted.groupby(keys, observed=True)

for l in lags:
    df[f'sold_lag_{l}'] = gs.shift(l).astype(np.float32)
for w in ROLL_WINDOWS:
    df[f'rolling_sold_mean_{w}'] = gs.transform(
        lambda x, w=w: x.rolling(w, min_periods=2).mean()).astype(np.float32)
    df[f'rolling_sold_std_{w}'] = gs.transform(
        lambda x, w=w: x.rolling(w, min_periods=2).std()).astype(np.float32)
df['expanding_sold_mean'] = gs.transform(
    lambda x: x.expanding(2).mean()).astype(np.float32)

## 6. Create lag and rolling features

Generate lagged demand, rolling means/standard deviations, and expanding means. The 28-day shift keeps the feature construction aligned with the forecasting information boundary.

In [ ]:
enc_specs = {
    'item_sold_avg': ['item_id'],
    'state_sold_avg': ['state_id'],
    'store_sold_avg': ['store_id'],
    'cat_sold_avg': ['cat_id'],
    'dept_sold_avg': ['dept_id'],
    'cat_dept_sold_avg': ['cat_id', 'dept_id'],
    'store_item_sold_avg': ['store_id', 'item_id'],
    'cat_item_sold_avg': ['cat_id', 'item_id'],
    'dept_item_sold_avg': ['dept_id', 'item_id'],
    'state_store_sold_avg': ['state_id', 'store_id'],
    'state_store_cat_sold_avg': ['state_id', 'store_id', 'cat_id'],
    'store_cat_dept_sold_avg': ['store_id', 'cat_id', 'dept_id'],
}
train_mask = df['d'] <= TRAIN_END
for name, keys_enc in enc_specs.items():
    m = df.loc[train_mask].groupby(keys_enc, observed=True)['y'].mean()
    if len(keys_enc) == 1:
        df[name] = df[keys_enc[0]].map(m).astype(np.float32)
    else:
        df[name] = pd.MultiIndex.from_frame(df[keys_enc]).map(m).astype(np.float32)

## 7. Create leakage-controlled aggregate encodings

Calculate group averages on the training period only, then map them to all rows. These encodings must not use validation or public/test targets.

In [ ]:
df = df[df['d'] >= START_TRAIN].reset_index(drop=True)

DROP = {'id', 'id_str', 'd', 'wm_yr_wk', 'sold', 'y', 'trend', 'store_id', 'dept_id'}
FEATURES = [c for c in df.columns if c not in DROP]
print('Số feature:', len(FEATURES))
print('Số dòng:', len(df))

## 8. Select the final training rows and feature list

Drop identifiers, target columns, and helper columns that are not model inputs. The resulting feature list must stay compatible with the saved LightGBM artifacts.

In [ ]:
def cv_score(rmses, lam=LAMBDA_STD):
    rmses = np.asarray(rmses, dtype=float)
    m, s = rmses.mean(), rmses.std(ddof=0)
    return m + lam * s, m, s

def cv_early_stop(names, patience, lam=LAMBDA_STD):
    state = {'best': np.inf, 'best_iter': 0, 'best_mean': np.nan, 'best_std': np.nan}

    def _cb(env):
        vals = [r[2] for r in env.evaluation_result_list if r[0] in names]
        score, m, s = cv_score(vals, lam)
        it = env.iteration
        if score < state['best']:
            state.update(best=score, best_iter=it, best_mean=m, best_std=s)
        elif it - state['best_iter'] >= patience:
            raise EarlyStopException(state['best_iter'], [])

    _cb.order = 30
    return _cb, state

## 9. Define cross-validation scoring and early stopping

Use the three 28-day validation windows to select a stable boosting round based on mean RMSE plus a variability penalty.

In [ ]:
MODEL_DIR_DT.mkdir(parents=True, exist_ok=True)
models_dt, rounds_dt, cv_stats = {}, {}, []

for s, dp in tqdm(groups):
    name = f'{name_maps["store_id"][s]}_{name_maps["dept_id"][dp]}'
    sdf = df[(df['store_id'] == s) & (df['dept_id'] == dp)]
    fit = sdf[sdf['d'] <= TRAIN_END]
    if len(fit) == 0:
        print('Bỏ qua:', name)
        continue

    train_set = lgb.Dataset(fit[FEATURES], label=fit['y'])
    valid_sets = []
    for k in VAL_NAMES:
        a, b = CV_WINDOWS[k]
        v = sdf[(sdf['d'] >= a) & (sdf['d'] <= b)]
        valid_sets.append(lgb.Dataset(v[FEATURES], label=v['y'], reference=train_set))

    cb, state = cv_early_stop(VAL_NAMES, PATIENCE)
    booster = lgb.train(PARAMS_DT, train_set, num_boost_round=MAX_ROUNDS,
                        valid_sets=valid_sets, valid_names=VAL_NAMES,
                        callbacks=[cb, lgb.log_evaluation(200)])

    rounds_dt[(s, dp)] = state['best_iter'] + 1
    models_dt[(s, dp)] = booster
    joblib.dump(booster, f'{MODEL_DIR_DT}/model_{name}.pkl')
    cv_stats.append({'group': name, 'best_round': rounds_dt[(s, dp)],
                     'mean_rmse': state['best_mean'], 'std_rmse': state['best_std'],
                     'score': state['best']})

    del train_set, valid_sets, sdf, fit
    gc.collect()

cv_stats = pd.DataFrame(cv_stats)
cv_stats.to_csv(DATA_DIR / 'cv_stats_detrend.csv', index=False)
print('Đã train:', len(models_dt), 'model')

## 10. Train and save one model per store/department group

Train each LightGBM Booster, apply the custom early stopping callback, and save model files as `Models/model_<store>_<department>.pkl`. CV summary is saved in `Data/cv_stats_detrend.csv`.

In [ ]:
def rmsse_scale(S, end_day):
    scales = np.full(S.shape[0], np.nan)
    for i in range(S.shape[0]):
        y = S[i, :end_day]
        nz = np.flatnonzero(y != 0)
        if len(nz) == 0:
            continue
        seg = y[nz[0]:]
        if len(seg) > 1:
            scales[i] = np.mean(np.diff(seg) ** 2)
    return scales

def rmsse_per_series(Y, P, scale):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.sqrt(np.mean((Y - P) ** 2, axis=1) / scale)

def predict_raw(a, b):
    W = df[(df['d'] >= a) & (df['d'] <= b)]
    parts = []
    for (s, dp), bst in models_dt.items():
        m = (W['store_id'] == s) & (W['dept_id'] == dp)
        part = W.loc[m, ['id_str', 'd', 'trend']].copy()
        part['pred'] = np.clip(
            bst.predict(W.loc[m, FEATURES], num_iteration=rounds_dt[(s, dp)])
            + part['trend'].to_numpy(), 0, None)
        parts.append(part[['id_str', 'd', 'pred']])
    pl = pd.concat(parts, ignore_index=True)
    return (pl.pivot(index='id_str', columns='d', values='pred')
            .reindex(index=ids, columns=range(a, b + 1)).to_numpy(dtype=np.float64))

## 11. Evaluate forecast accuracy

Generate predictions for each validation/public window and calculate RMSSE statistics. These results are diagnostics only; they do not change the inventory policy parameters.

In [ ]:
rows, P_dt = [], {}
for wname, (a, b) in {**CV_WINDOWS, 'public': PUBLIC}.items():
    P = predict_raw(a, b)
    P_dt[wname] = P
    scale = rmsse_scale(S_sold, a - 1)
    r = rmsse_per_series(S_sold[:, a - 1:b], P, scale)
    rv = r[np.isfinite(r)]
    rows.append({'window': wname, 'RMSSE mean': rv.mean(),
                 'RMSSE median': np.median(rv), 'n_series': len(rv)})

res_dt = pd.DataFrame(rows)
print('Kết quả luồng detrend:')
print(res_dt.round(4).to_string(index=False))
res_dt.to_csv(DATA_DIR / 'rmsse_detrend.csv', index=False)

## 12. Save the processed feature table and metadata

Write `Data/df_detrend.parquet` and `Data/name_maps.json`. On Kaggle, download the generated `inventory_optimization` folder and copy its `Data/` and `Models/` folders into the application project.

In [ ]:
import os

os.makedirs(OUT, exist_ok=True)
path_pq = DATA_DIR / 'df_detrend.parquet'

df.to_parquet(path_pq, index=False, compression='snappy')

with open(DATA_DIR / 'name_maps.json', 'w', encoding='utf-8') as fh:
    json.dump(name_maps, fh, ensure_ascii=False)

print('Saved processed data:', path_pq)
print('Saved metadata:', DATA_DIR / 'name_maps.json')
print('Saved models:', MODEL_DIR_DT)
print('Đã lưu:', path_pq)
print('Shape:', df.shape, '| dung lượng: %.1f MB' % (os.path.getsize(path_pq) / 1024**2))